# TP6 — Couche Gold : feature engineering complet

Construit `gold_machine_hourly_feature` à partir de la couche Silver.
1 ligne = 1 machine × 1 heure | ~91 colonnes | prêt pour sklearn / LightGBM

**Familles de features :**
- Agrégats 1h bruts
- Rolling 6h / 12h / 24h (mean, max, std) — symétrique sur les 4 signaux
- Tendances : `delta_1h`, `delta_3h`, `trend_6h`
- Z-scores : 24h glissant + baseline machine (train-only, sans leakage)
- Production : `pieces_produced_sum_24h`, `capacity_utilization_pct`
- Incidents lookback : 24h et 7j, `hours_since_last_incident`, 9 types
- Maintenance lookback : `days_since_last_maintenance`, `maintenance_count_prev_30d`
- Labels : `label_failure_next_6h/12h/24h/48h` + `future_incident_count_6h/12h/24h/48h` (compte brut, exclu des features — fuite)
- Split temporel : train 70 % / validation 15 % / test 15 %


## 0. Connexion

In [1]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
import pandas as pd
import numpy as np
from datetime import datetime, timezone

db_url = URL.create(
    drivername='postgresql+psycopg2',
    username='indusense_user',
    password='ThEP@ssW0rd',
    host='localhost',
    port=5432,
    database='indusense_db',
)
engine = create_engine(db_url)
with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar())


PostgreSQL 15.1 (Debian 15.1-1.pgdg110+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 10.2.1-6) 10.2.1 20210110, 64-bit


## 1. Lecture Silver

In [2]:
df_sensors = pd.read_sql(text("""
    SELECT machine_id, observed_at, sensor_type, sensor_value
    FROM silver_sensor_reading
    WHERE is_missing = false AND is_duplicate = false
"""), engine, parse_dates=['observed_at'])

df_inc = pd.read_sql(text("""
    SELECT si.machine_id, si.occurred_at, si.severity, si.is_label_event,
           bi.type_surchauffe, bi.type_baisse_pression, bi.type_vibration,
           bi.type_bruit_mecanique, bi.type_surconsommation, bi.type_blocage_mecanique,
           bi.type_alarme_capteur, bi.type_arret_urgence, bi.type_defaut_qualite
    FROM silver_incident si
    JOIN bronze_incidents bi ON bi.incident_id = si.incident_code
"""), engine, parse_dates=['occurred_at'])

df_maint = pd.read_sql(text("""
    SELECT machine_id, performed_at
    FROM silver_maintenance
"""), engine, parse_dates=['performed_at'])

df_machine = pd.read_sql(text("""
    SELECT machine_code AS machine_id, max_hourly_capacity_pieces
    FROM machine
"""), engine)

print(f'Capteurs : {len(df_sensors):,} lignes')
print(f'Incidents: {len(df_inc):,} lignes')
print(f'Maintenance: {len(df_maint):,} lignes')
print(f'Machines (referentiel capacite): {len(df_machine):,} lignes')
print(f'Période  : {df_sensors["observed_at"].min()} → {df_sensors["observed_at"].max()}')


Capteurs : 661,920 lignes
Incidents: 1,245 lignes
Maintenance: 1,562 lignes
Machines (referentiel capacite): 15 lignes
Période  : 2025-06-01 00:00:00+00:00 → 2026-06-08 23:00:00+00:00


## 2. Pivot → format large (1 ligne par machine × heure)

In [3]:
df_wide = df_sensors.pivot_table(
    index=['machine_id', 'observed_at'],
    columns='sensor_type', values='sensor_value', aggfunc='mean'
).reset_index()
df_wide.columns.name = None
for col in ['temperature_c','pressure_bar','voltage_mean_v','rotation_mean_rpm','pieces_produced']:
    if col not in df_wide.columns:
        df_wide[col] = float('nan')
df_wide = df_wide.sort_values(['machine_id','observed_at']).reset_index(drop=True)
print(f'Format large : {len(df_wide):,} lignes × {len(df_wide.columns)} colonnes')


Format large : 132,940 lignes × 7 colonnes


## 3. Rolling features par machine

Fenêtres **passées** (closed='left') pour éviter tout leakage sur la valeur courante.


In [4]:
SIGNALS = {
    'temp'     : 'temperature_c',
    'pressure' : 'pressure_bar',
    'voltage'  : 'voltage_mean_v',
    'rotation' : 'rotation_mean_rpm',
}
WINDOWS = {'1h': '1h', '6h': '6h', '12h': '12h', '24h': '24h'}

chunks = []
for machine, grp in df_wide.groupby('machine_id'):
    grp = grp.sort_values('observed_at').copy()
    idx = grp.set_index('observed_at')

    # Agrégats 1h bruts
    grp['temp_mean_1h']           = grp['temperature_c']
    grp['temp_max_1h']            = grp['temperature_c']
    grp['temp_std_1h']            = float('nan')
    grp['pressure_mean_1h']       = grp['pressure_bar']
    grp['pressure_max_1h']        = grp['pressure_bar']
    grp['pressure_std_1h']        = float('nan')
    grp['voltage_mean_1h']        = grp['voltage_mean_v']
    grp['rotation_mean_1h']       = grp['rotation_mean_rpm']
    grp['pieces_produced_sum_1h'] = pd.to_numeric(grp['pieces_produced'], errors='coerce')

    # Rolling 6h / 12h / 24h
    for w in ['6h', '12h', '24h']:
        roll = idx[list(SIGNALS.values())].rolling(window=w, closed='left', min_periods=1)
        prefix_map = {'temperature_c': 'temp', 'pressure_bar': 'pressure',
                      'voltage_mean_v': 'voltage', 'rotation_mean_rpm': 'rotation'}
        for col, pfx in prefix_map.items():
            grp[f'{pfx}_mean_{w}'] = roll[col].mean().values
            grp[f'{pfx}_std_{w}']  = roll[col].std().values
            grp[f'{pfx}_max_{w}']  = roll[col].max().values

    # Production 24h
    grp['pieces_produced_sum_24h'] = (
        idx['pieces_produced'].rolling(window='24h', closed='left', min_periods=1).sum().values
    )

    # Tendances : delta_1h, delta_3h, trend_6h
    for col, pfx in prefix_map.items():
        s = grp[col]
        grp[f'{pfx}_delta_1h']  = s.diff(1)
        grp[f'{pfx}_delta_3h']  = s.diff(3)
        grp[f'{pfx}_trend_6h']  = s.diff(6)

    chunks.append(grp)

df = pd.concat(chunks, ignore_index=True)
print(f'Rolling features : {len(df):,} lignes × {len(df.columns)} colonnes')


Rolling features : 132,940 lignes × 65 colonnes


## 3bis. Taux d'utilisation de la capacité — `capacity_utilization_pct`

Ratio entre la production horaire réelle et la capacité horaire nominale de la machine (référentiel `machine.sql`). Donnée statique par machine — simple jointure, pas de fuite possible (aucune dépendance temporelle).

In [5]:
df = df.merge(df_machine, on='machine_id', how='left')
df['capacity_utilization_pct'] = (
    df['pieces_produced_sum_1h'] / df['max_hourly_capacity_pieces'] * 100
)
df.drop(columns=['max_hourly_capacity_pieces'], inplace=True)

print('capacity_utilization_pct calculé')
print(df['capacity_utilization_pct'].describe())


capacity_utilization_pct calculé
count    132940.000000
mean         75.221322
std          32.349370
min           0.000000
25%          45.945946
50%          91.666667
75%         101.754386
max         134.000000
Name: capacity_utilization_pct, dtype: float64


## 4. Z-scores

### 4a. Z-score glissant 24h
`(valeur_courante - mean_24h) / std_24h`


In [6]:
df['temp_zscore_24h']     = (df['temperature_c'] - df['temp_mean_24h']) / df['temp_std_24h'].replace(0, float('nan'))
df['pressure_zscore_24h'] = (df['pressure_bar']   - df['pressure_mean_24h']) / df['pressure_std_24h'].replace(0, float('nan'))
print('Z-score 24h calculé')


Z-score 24h calculé


### 4b. Split temporel préliminaire (nécessaire pour le z-score machine)

**Règle anti-leakage :** la baseline machine est calculée uniquement sur le train set,
puis appliquée à val et test tels quels.


In [7]:
dates  = df['observed_at'].sort_values().unique()
n      = len(dates)
cut_70 = dates[int(n * 0.70)]
cut_85 = dates[int(n * 0.85)]

df['split_set'] = np.where(
    df['observed_at'] < cut_70, 'train',
    np.where(df['observed_at'] < cut_85, 'validation', 'test')
)
print(df['split_set'].value_counts())


split_set
train         93079
test          19944
validation    19917
Name: count, dtype: int64


### 4c. Z-score machine (baseline train-only)

In [8]:
train_stats = (
    df[df['split_set'] == 'train']
    .groupby('machine_id')[['temperature_c', 'pressure_bar']]
    .agg(['mean', 'std'])
)
train_stats.columns = ['temp_train_mean','temp_train_std','pressure_train_mean','pressure_train_std']

df = df.merge(train_stats.reset_index(), on='machine_id', how='left')
df['temp_zscore_machine']     = (df['temperature_c'] - df['temp_train_mean']) / df['temp_train_std'].replace(0, float('nan'))
df['pressure_zscore_machine'] = (df['pressure_bar']  - df['pressure_train_mean']) / df['pressure_train_std'].replace(0, float('nan'))
df.drop(columns=['temp_train_mean','temp_train_std','pressure_train_mean','pressure_train_std'], inplace=True)
print('Z-score machine calculé (leakage-free)')


Z-score machine calculé (leakage-free)

## 5. Features incidents lookback

Lookback 24h, 7j, `hours_since_last_incident`, et comptage par type d'incident.


In [9]:
TYPE_COLS = [
    'type_surchauffe','type_baisse_pression','type_vibration','type_bruit_mecanique',
    'type_surconsommation','type_blocage_mecanique','type_alarme_capteur',
    'type_arret_urgence','type_defaut_qualite'
]
inc_cols = ['incident_count_prev_24h','incident_max_severity_prev_24h',
            'incident_count_prev_7d','hours_since_last_incident'] + \
           [f'{t}_count_prev_24h' for t in TYPE_COLS]
for col in inc_cols:
    df[col] = 0
df['hours_since_last_incident'] = float('nan')

for machine in df['machine_id'].unique():
    mask_f = df['machine_id'] == machine
    hours  = df.loc[mask_f, 'observed_at']
    evts   = df_inc[df_inc['machine_id'] == machine].sort_values('occurred_at')
    if evts.empty:
        continue
    occ = evts['occurred_at'].values
    for idx, h in hours.items():
        w24  = evts[(evts['occurred_at'] >= h - pd.Timedelta(hours=24)) & (evts['occurred_at'] < h)]
        w7d  = evts[(evts['occurred_at'] >= h - pd.Timedelta(days=7))  & (evts['occurred_at'] < h)]
        past = occ[occ < np.datetime64(h)]
        df.at[idx, 'incident_count_prev_24h']        = len(w24)
        df.at[idx, 'incident_max_severity_prev_24h'] = int(w24['severity'].max()) if len(w24) else 0
        df.at[idx, 'incident_count_prev_7d']         = len(w7d)
        df.at[idx, 'hours_since_last_incident']      = (
            (np.datetime64(h) - past[-1]) / np.timedelta64(1, 'h') if len(past) else float('nan')
        )
        for t in TYPE_COLS:
            df.at[idx, f'{t}_count_prev_24h'] = int(w24[t].sum()) if len(w24) else 0

print('Incident features calculées')
print('incident_count_prev_24h > 0 :', (df['incident_count_prev_24h'] > 0).sum())


C:\Users\Aelion\AppData\Local\Temp\ipykernel_27440\2402631515.py:23: UserWarning: no explicit representation of timezones available for np.datetime64
  past = occ[occ < np.datetime64(h)]
C:\Users\Aelion\AppData\Local\Temp\ipykernel_27440\2402631515.py:28: UserWarning: no explicit representation of timezones available for np.datetime64
  (np.datetime64(h) - past[-1]) / np.timedelta64(1, 'h') if len(past) else float('nan')


Incident features calculées
incident_count_prev_24h > 0 : 22343


## 5bis. Features maintenance lookback

Même principe que le lookback incidents (§5) : `days_since_last_maintenance` (dernière intervention avant l'heure courante) et `maintenance_count_prev_30d` (nombre d'interventions sur les 30 jours précédents), calculés **intra-machine**, sans regarder le futur.

In [10]:
maint_cols = ['days_since_last_maintenance', 'maintenance_count_prev_30d']
for col in maint_cols:
    df[col] = 0
df['days_since_last_maintenance'] = float('nan')

for machine in df['machine_id'].unique():
    mask_f = df['machine_id'] == machine
    hours  = df.loc[mask_f, 'observed_at']
    evts   = df_maint[df_maint['machine_id'] == machine].sort_values('performed_at')
    if evts.empty:
        continue
    occ = evts['performed_at'].values
    for idx, h in hours.items():
        w30d = evts[(evts['performed_at'] >= h - pd.Timedelta(days=30)) & (evts['performed_at'] < h)]
        past = occ[occ < np.datetime64(h)]
        df.at[idx, 'maintenance_count_prev_30d'] = len(w30d)
        df.at[idx, 'days_since_last_maintenance'] = (
            (np.datetime64(h) - past[-1]) / np.timedelta64(1, 'D') if len(past) else float('nan')
        )

print('Maintenance features calculées')
n_na = df['days_since_last_maintenance'].isna().sum()
print(f'days_since_last_maintenance NaN (pas encore de maintenance passée) : {n_na:,} / {len(df):,}')
print(df[['days_since_last_maintenance', 'maintenance_count_prev_30d']].describe())


C:\Users\Aelion\AppData\Local\Temp\ipykernel_27440\3063861615.py:15: UserWarning: no explicit representation of timezones available for np.datetime64
  past = occ[occ < np.datetime64(h)]
C:\Users\Aelion\AppData\Local\Temp\ipykernel_27440\3063861615.py:18: UserWarning: no explicit representation of timezones available for np.datetime64
  (np.datetime64(h) - past[-1]) / np.timedelta64(1, 'D') if len(past) else float('nan')


Maintenance features calculées
days_since_last_maintenance NaN (pas encore de maintenance passée) : 3,418 / 132,940
       days_since_last_maintenance  maintenance_count_prev_30d
count                129522.000000               132940.000000
mean                      9.254924                    8.017233
std                      10.057190                    7.654607
min                       0.002083                    0.000000
25%                       1.948611                    3.000000
50%                       5.822222                    6.000000
75%                      12.926389                   11.000000
max                      60.083333                   47.000000


## 6. Labels multi-horizons

Technique du **lookahead inversé** : pour chaque label event à t,
on marque toutes les heures dans [t-H, t) comme `True` pour l'horizon H.


In [11]:
for col in ['label_failure_next_6h','label_failure_next_12h',
            'label_failure_next_24h','label_failure_next_48h']:
    df[col] = False

for col in ['future_incident_count_6h','future_incident_count_12h',
            'future_incident_count_24h','future_incident_count_48h']:
    df[col] = 0

label_events = df_inc[df_inc['is_label_event']]
HORIZONS = [
    ('label_failure_next_6h',  'future_incident_count_6h',  6),
    ('label_failure_next_12h', 'future_incident_count_12h', 12),
    ('label_failure_next_24h', 'future_incident_count_24h', 24),
    ('label_failure_next_48h', 'future_incident_count_48h', 48),
]
for _, evt in label_events.iterrows():
    for label_col, count_col, h in HORIZONS:
        mask = (
            (df['machine_id'] == evt['machine_id']) &
            (df['observed_at'] >= evt['occurred_at'] - pd.Timedelta(hours=h)) &
            (df['observed_at'] <  evt['occurred_at'])
        )
        df.loc[mask, label_col] = True
        df.loc[mask, count_col] += 1

for label_col, count_col, _ in HORIZONS:
    n = df[label_col].sum()
    print(f'{label_col}: {n:,} True ({n/len(df)*100:.1f}%)  |  {count_col} max={df[count_col].max()}')


label_failure_next_6h: 1,212 True (0.9%)  |  future_incident_count_6h max=2
label_failure_next_12h: 2,406 True (1.8%)  |  future_incident_count_12h max=2
label_failure_next_24h: 4,745 True (3.6%)  |  future_incident_count_24h max=2
label_failure_next_48h: 9,157 True (6.9%)  |  future_incident_count_48h max=3


## 7. Batch Gold + ingestion

In [12]:
started_at = datetime.now(timezone.utc)
with engine.begin() as conn:
    batch_gold = conn.execute(text("""
        INSERT INTO ingestion_batch (source_name, source_file, started_at, status)
        VALUES ('gold', 'silver_sensor_reading + silver_incident + silver_maintenance', :ts, 'running')
        RETURNING ingestion_batch_id
    """), {'ts': started_at}).scalar()
df['ingestion_batch_id'] = str(batch_gold)
df['window_start'] = df['observed_at']
df['window_end']   = df['observed_at'] + pd.Timedelta(hours=1)
print(f'Batch Gold : {batch_gold}')


Batch Gold : 41cf3e53-a272-4e8e-8530-e6f9794b49d7


In [13]:
GOLD_COLS = [
    'ingestion_batch_id','machine_id','window_start','window_end','split_set',
    # 1h
    'temp_mean_1h','temp_max_1h','temp_std_1h',
    'pressure_mean_1h','pressure_max_1h','pressure_std_1h',
    'voltage_mean_1h','rotation_mean_1h','pieces_produced_sum_1h',
    # 6h
    'temp_mean_6h','temp_max_6h','temp_std_6h',
    'pressure_mean_6h','pressure_max_6h','pressure_std_6h',
    'voltage_mean_6h','voltage_max_6h','voltage_std_6h',
    'rotation_mean_6h','rotation_max_6h','rotation_std_6h',
    # 12h
    'temp_mean_12h','temp_max_12h','temp_std_12h',
    'pressure_mean_12h','pressure_max_12h','pressure_std_12h',
    'voltage_mean_12h','voltage_max_12h','voltage_std_12h',
    'rotation_mean_12h','rotation_max_12h','rotation_std_12h',
    # 24h
    'temp_mean_24h','temp_max_24h','temp_std_24h',
    'pressure_mean_24h','pressure_max_24h','pressure_std_24h',
    'voltage_mean_24h','voltage_max_24h','voltage_std_24h',
    'rotation_mean_24h','rotation_max_24h','rotation_std_24h',
    # Tendances
    'temp_delta_1h','temp_delta_3h','temp_trend_6h',
    'pressure_delta_1h','pressure_delta_3h','pressure_trend_6h',
    'voltage_delta_1h','voltage_delta_3h','voltage_trend_6h',
    'rotation_delta_1h','rotation_delta_3h','rotation_trend_6h',
    # Z-scores
    'temp_zscore_24h','pressure_zscore_24h',
    'temp_zscore_machine','pressure_zscore_machine',
    # Production
    'pieces_produced_sum_24h','capacity_utilization_pct',
    # Incidents
    'incident_count_prev_24h','incident_max_severity_prev_24h',
    'incident_count_prev_7d','hours_since_last_incident',
    'type_surchauffe_count_prev_24h','type_baisse_pression_count_prev_24h',
    'type_vibration_count_prev_24h','type_bruit_mecanique_count_prev_24h',
    'type_surconsommation_count_prev_24h','type_blocage_mecanique_count_prev_24h',
    'type_alarme_capteur_count_prev_24h','type_arret_urgence_count_prev_24h',
    'type_defaut_qualite_count_prev_24h',
    # Maintenance
    'days_since_last_maintenance','maintenance_count_prev_30d',
    # Labels
    'label_failure_next_6h','label_failure_next_12h',
    'label_failure_next_24h','label_failure_next_48h',
    # Compte brut d'incidents futurs (avant seuillage)
    'future_incident_count_6h','future_incident_count_12h',
    'future_incident_count_24h','future_incident_count_48h',
]

with engine.begin() as conn:
    conn.execute(text('TRUNCATE TABLE gold_machine_hourly_feature RESTART IDENTITY'))

df[GOLD_COLS].to_sql(
    'gold_machine_hourly_feature', engine,
    if_exists='append', index=False, method='multi', chunksize=2000
)
print(f'gold_machine_hourly_feature : {len(df):,} lignes | {len(GOLD_COLS)} colonnes')


gold_machine_hourly_feature : 132,940 lignes | 91 colonnes


## 8. Clôture batch

In [14]:
finished_at = datetime.now(timezone.utc)
with engine.begin() as conn:
    conn.execute(text("""
        UPDATE ingestion_batch
        SET finished_at=:fin, rows_read=:r, rows_loaded=:r, rows_rejected=0, status='done'
        WHERE ingestion_batch_id=:bid
    """), {'fin': finished_at, 'r': len(df), 'bid': str(batch_gold)})
print('Batch Gold clôturé')


Batch Gold clôturé


## 9. Vérification finale

In [15]:
display(pd.read_sql(text("""
    SELECT source_name, rows_read, rows_loaded, status
    FROM ingestion_batch WHERE source_name='gold'
    ORDER BY started_at DESC LIMIT 1
"""), engine))

print('--- Distribution par split ---')
display(pd.read_sql(text("""
    SELECT split_set,
           COUNT(*)                     AS nb_lignes,
           COUNT(DISTINCT machine_id)   AS nb_machines,
           MIN(window_start)            AS debut,
           MAX(window_start)            AS fin,
           ROUND(AVG(label_failure_next_24h::int)::numeric*100,1) AS pct_label_24h,
           ROUND(AVG(label_failure_next_6h::int)::numeric*100,1)  AS pct_label_6h,
           ROUND(AVG(days_since_last_maintenance)::numeric,2)     AS avg_days_since_maint,
           ROUND(AVG(maintenance_count_prev_30d)::numeric,2)      AS avg_maint_30d
    FROM gold_machine_hourly_feature
    GROUP BY split_set ORDER BY debut
"""), engine))

print('--- Apercu features ---')
display(pd.read_sql(text("""
    SELECT machine_id, window_start,
           temp_mean_1h, temp_mean_24h, temp_zscore_24h, temp_zscore_machine,
           temp_delta_1h, temp_trend_6h,
           incident_count_prev_24h, days_since_last_maintenance, maintenance_count_prev_30d,
           label_failure_next_24h, split_set
    FROM gold_machine_hourly_feature
    ORDER BY window_start LIMIT 5
"""), engine))


,source_name,rows_read,rows_loaded,status
0,gold,132940,132940,done


--- Distribution par split ---


,split_set,nb_lignes,nb_machines,debut,fin,pct_label_24h,pct_label_6h,avg_days_since_maint,avg_maint_30d
0,train,93079,15,2025-06-01 00:00:00+00:00,2026-02-17 01:00:00+00:00,3.6,0.9,9.14,7.80
1,validation,19917,15,2026-02-17 02:00:00+00:00,2026-04-14 00:00:00+00:00,3.4,0.8,9.72,8.72
2,test,19944,15,2026-04-14 01:00:00+00:00,2026-06-08 23:00:00+00:00,3.6,0.9,9.29,8.31


--- Apercu features ---


,machine_id,window_start,temp_mean_1h,temp_mean_24h,temp_zscore_24h,temp_zscore_machine,temp_delta_1h,temp_trend_6h,incident_count_prev_24h,days_since_last_maintenance,maintenance_count_prev_30d,label_failure_next_24h,split_set
0,MACH-11,2025-06-01 00:00:00+00:00,38.000,None,None,-2.040276,None,None,0,None,0,False,train
1,MACH-13,2025-06-01 00:00:00+00:00,42.622,None,None,-1.840287,None,None,0,None,0,False,train
2,MACH-15,2025-06-01 00:00:00+00:00,47.144,None,None,-0.497162,None,None,0,None,0,False,train
3,MACH-08,2025-06-01 00:00:00+00:00,43.398,None,None,-1.401437,None,None,0,None,0,False,train
4,MACH-05,2025-06-01 00:00:00+00:00,50.395,None,None,0.129209,None,None,0,None,0,False,train
